# Week 4: Vector Power-Up & Raster Integration

**Student Worksheet** — Fill in the code cells using AI assistance or your own code.

This week you will:
1. Master vector aggregation (dissolve & groupby)
2. Understand raster data as NumPy arrays with spatial metadata
3. Compute terrain slope from DEM
4. Extract raster values into vector shapes (zonal statistics)

**Packages needed:** `geopandas`, `rioxarray`, `rasterstats`, `numpy`, `matplotlib`

> If you haven't installed them yet, run:  
> `pip install rioxarray rasterstats geopandas folium mapclassify matplotlib`

## Cell [1] — Environment Setup

Import all necessary packages and load Week 3 township data.

In [ ]:
# 導入所需套件
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# import rioxarray as rxr  # 後續需要時再導入
# import xarray as xr      # 後續需要時再導入
from urllib.parse import quote

# 載入 TGOS 鄉鎮資料
TGOS_BASE = 'https://www.tgos.tw/tgos/VirtualDir/Product/3fe61d4a-ca23-4f45-8aca-4a536f40f290/'
url = TGOS_BASE + quote('鄉(鎮、市、區)界線1140318.zip')

# 載入圖層
townships = gpd.read_file(url, layer='TOWN_MOI_1140318')

# 轉換為 EPSG:3826
townships = townships.to_crs('EPSG:3826')

# 印出資訊
print(f"資料形狀: {townships.shape}")
print(f"座標系統: {townships.crs}")
print("\n前5行資料:")
print(townships.head())

In [ ]:
import geopandas as gpd
import numpy as np

print(np.__version__)

## Cell [2] — Dissolve & Groupby

**Dissolve**: Merge 368 townships → ~22 counties  
**Groupby**: Count shelters per county and sum capacity

In [ ]:
# Cell [2]: YOUR CODE HERE
# 1. Dissolve townships by 'COUNTYNAME' → counties GeoDataFrame
# 2. Print: how many counties? What geometry types?
# 3. Calculate area in km² for each county
# 4. Show the top 5 largest counties

# AI Prompt suggestion:
# "Dissolve my townships GeoDataFrame by COUNTYNAME to create county boundaries.
#  Calculate each county's area in km². Show the top 5 largest."
# 將鄉鎮按 'COUNTYNAME' 進行 dissolve → 縣市 GeoDataFrame
counties = townships.dissolve(by='COUNTYNAME')
 
# 印出：有多少個縣市？幾何類型是什麼？
print(f"縣市數量: {len(counties)}")
print(f"幾何類型: {counties.geometry.type.unique()}")
 
# 計算每個縣市的面積（平方公里）
counties['area_km2'] = counties.geometry.area / 1_000_000  # 轉換為平方公里
 
# 顯示面積最大的前5個縣市
print("\n面積最大的前5個縣市:")
print(counties[['area_km2']].sort_values('area_km2', ascending=False).head())

---

## 🔬 Lab 1: Vector Aggregation (20 minutes)

**Goal**: Practice dissolve & groupby before entering the raster world.

> **⚠️ 注意**：Lab 1 使用**合成資料**（synthetic data）練習 dissolve/groupby 操作。課後作業（ARIA v2.0）會使用**真實的避難收容所 CSV**，操作方式相同但資料來源不同。

### Step 1: Dissolve townships → counties

In [ ]:
# Lab 1 Step 1: YOUR CODE HERE
# Dissolve all townships into counties using COUNTYNAME
# Verify: pick Hualien County (花蓮縣) and check its geometry type and area

# AI Prompt suggestion:
# "Dissolve my townships GeoDataFrame by 'COUNTYNAME'.
#  Then filter for '花蓮縣' and print its geometry type and area in km²."
# 將所有鄉鎮按 COUNTYNAME 進行 dissolve 來建立縣市
counties = townships.dissolve(by='COUNTYNAME')

# 驗證：選擇花蓮縣並檢查其幾何類型和面積
hualien = counties.loc['花蓮縣']
print(f"花蓮縣幾何類型: {hualien.geometry.type}")
print(f"花蓮縣面積: {hualien.geometry.area / 1_000_000:.2f} 平方公里")

### Step 2: Create synthetic shelter data for Hualien

In [ ]:
# Lab 1 Step 2: YOUR CODE HERE
# Create synthetic shelter points in Hualien County
# Each town should have 2-8 shelters with random capacity (50-500)
# Place shelters at each town's centroid (simplified)

# AI Prompt suggestion:
# "Filter my townships for '花蓮縣' towns. For each town, create 2-8 random shelters
#  at the town centroid with random capacity 50-500.
#  Use np.random.seed(42) for reproducibility.
#  Create a GeoDataFrame with columns: shelter_id, TOWNNAME, capacity, geometry.
#  CRS should be EPSG:3826."
# 設定隨機種子以確保結果可重現
np.random.seed(42)

# 篩選花蓮縣的鄉鎮
hualien_towns = townships[townships['COUNTYNAME'] == '花蓮縣'].copy()

# 創建避難收容所資料
shelters = []
shelter_id = 1

for _, town in hualien_towns.iterrows():
    # 每個鄉鎮創建 2-8 個避難收容所
    num_shelters = np.random.randint(2, 9)
    
    for i in range(num_shelters):
        # 在鄉鎮中心點創建避難收容所
        capacity = np.random.randint(50, 501)
        shelters.append({
            'shelter_id': shelter_id,
            'TOWNNAME': town['TOWNNAME'],
            'capacity': capacity,
            'geometry': town.geometry.centroid
        })
        shelter_id += 1

# 創建 GeoDataFrame
shelters_gdf = gpd.GeoDataFrame(shelters, crs='EPSG:3826')

print(f"花蓮縣共創建了 {len(shelters_gdf)} 個避難收容所")
print(f"分布於 {shelters_gdf['TOWNNAME'].nunique()} 個鄉鎮")
print("\n前5筆資料:")
print(shelters_gdf.head())

### Step 3: Groupby — Statistics per town

In [ ]:
# Lab 1 Step 3: YOUR CODE HERE
# Group shelters by TOWNNAME
# Calculate: count, total capacity, average capacity
# Sort by total capacity descending

# AI Prompt suggestion:
# "Group my shelters GeoDataFrame by 'TOWNNAME'.
#  For each town, calculate shelter_count, total_capacity (sum), and avg_capacity (mean).
#  Sort by total_capacity descending and display all rows."
# 按 TOWNNAME 分組避難收容所
town_summary = shelters_gdf.groupby('TOWNNAME').agg({
    'shelter_id': 'count',        # 避難收容所數量
    'capacity': ['sum', 'mean']   # 總容量和平均容量
}).round(2)

# 重新命名欄位
town_summary.columns = ['shelter_count', 'total_capacity', 'avg_capacity']

# 按總容量降序排列
town_summary = town_summary.sort_values('total_capacity', ascending=False)

print("花蓮縣各鄉鎮避難收容所統計:")
print(town_summary)


### Step 4: Merge stats back to geometry & visualize

In [ ]:
# Lab 1 Step 4: YOUR CODE HERE
# 1. Merge the groupby statistics back to the Hualien township geometry
# 2. Create an interactive map with .explore() showing total_capacity by color
# 3. Save the map as HTML (for Windsurf/VS Code users)

# AI Prompt suggestion:
# "Merge my town_summary DataFrame back into the Hualien townships GeoDataFrame
#  using TOWNNAME. Then create an .explore() map colored by total_capacity
#  using 'YlOrRd' colormap. Add tooltip showing TOWNNAME, shelter_count, total_capacity.
#  Remember to convert to EPSG:4326 for .explore().
#  Also save the map as 'outputs/lab1_hualien_capacity.html' using m.save()."
# 1. 將統計資料合併回花蓮鄉鎮幾何資料
hualien_towns_with_stats = hualien_towns.merge(town_summary, 
                                              left_on='TOWNNAME', 
                                              right_index=True,
                                              how='left')

# 2. 轉換為 WGS84 用於 explore()
hualien_towns_wgs84 = hualien_towns_with_stats.to_crs('EPSG:4326')

# 3. 創建互動式地圖
m = hualien_towns_wgs84.explore(
    column='total_capacity',
    cmap='YlOrRd',
    tooltip=['TOWNNAME', 'shelter_count', 'total_capacity', 'avg_capacity'],
    popup=True,
    legend=True,
    style_kwds={'color': 'black', 'weight': 1, 'fillOpacity': 0.7}
)

# 4. 顯示地圖
display(m)

# 5. 創建 outputs 目錄（如果不存在）
import os
if not os.path.exists('outputs'):
    os.makedirs('outputs')

# 6. 保存地圖為 HTML
m.save('outputs/lab1_hualien_capacity.html')
print("地圖已保存為 outputs/lab1_hualien_capacity.html")


---

## Cell [3] — The Raster Glass Box: It's Just a Matrix

A raster (like a DEM) is a **NumPy array with a GPS**. Each pixel = one numeric value.

In [ ]:
# Cell [3]: YOUR CODE HERE
# Create a synthetic 100x80 DEM with rioxarray
# Use valley profile: low center, mountains on sides
# Set CRS to EPSG:3826
# Print: shape, CRS, resolution, min/max elevation, top-left 5x5 values

# AI Prompt suggestion:
# "Create a synthetic 100x80 DEM simulating a valley (low center, high sides).
#  Elevation formula: 50 + 800*(2*X-1)^2 + 200*Y + 30*sin(8X)*cos(6Y)
#  where X and Y are normalized 0-1 meshgrids.
#  Wrap as xarray DataArray with rioxarray, set CRS EPSG:3826.
#  Use bounds: x=[302000, 303600], y=[2638000, 2640000].
#  Print shape, CRS, resolution, min/max, and the top-left 5x5 pixel values."
# 重新導入 rioxarray 和 xarray
# 創建合成 DEM（不使用 rioxarray）
import numpy as np
import matplotlib.pyplot as plt

# 設定網格
nx, ny = 100, 80
x = np.linspace(0, 1, nx)
y = np.linspace(0, 1, ny)
X, Y = np.meshgrid(x, y)

# 計算高程公式：谷地剖面
elevation = 50 + 800*(2*X-1)**2 + 200*Y + 30*np.sin(8*X)*np.cos(6*Y)

# 印出資訊
print(f"DEM 形狀: {elevation.shape}")
print(f"高程範圍: {elevation.min():.1f} - {elevation.max():.1f} 公尺")
print(f"左上角 5x5 像素值:")
print(elevation[:5, :5].round(1))

# 設定邊界資訊（模擬 EPSG:3826）
bounds = {'x': [302000, 303600], 'y': [2638000, 2640000]}
resolution_x = (bounds['x'][1] - bounds['x'][0]) / nx
resolution_y = (bounds['y'][1] - bounds['y'][0]) / ny
print(f"模擬解析度: {resolution_x:.1f} x {resolution_y:.1f} 公尺")
print(f"模擬座標系統: EPSG:3826")

# 儲存為變數供後續使用
dem_synth = elevation

## Cell [4] — Affine Transform

How does `Matrix[row, col]` become real-world `(X, Y)` coordinates?

In [ ]:
# Cell [4]: YOUR CODE HERE
# Read the affine transform from your DEM
# Print the 6 parameters and explain what each means
# Manually convert pixel [5, 10] to real-world coordinates

# AI Prompt suggestion:
# "Get the affine transform from my DEM (dem.rio.transform()).
#  Print each of the 6 parameters with labels (pixel width, rotation, top-left coords).
#  Then manually compute the real-world coordinates of pixel [5, 10]
#  using: real_x = a*col + b*row + c, real_y = d*col + e*row + f.
#  Also print the elevation value at that pixel."
# 創建仿射變換（模擬）
# 仿射變換的 6 個參數：[a, b, c, d, e, f]
# a = pixel width, b = rotation_x, c = top-left x
# d = rotation_y, e = pixel height (negative), f = top-left y

# 根據我們設定的邊界和解析度計算仿射變換
pixel_width = 16.0      # x 方向像素解析度
pixel_height = -25.0   # y 方向像素解析度（負值，因為 y 軸向下）
top_left_x = 302000    # 左上角 x 座標
top_left_y = 2640000   # 左上角 y 座標

# 仿射變換參數（假設無旋轉）
affine_params = [pixel_width, 0, top_left_x, 0, pixel_height, top_left_y]

print("仿射變換的 6 個參數:")
print(f"a (pixel width): {affine_params[0]} 公尺")
print(f"b (rotation x): {affine_params[1]} (無旋轉)")
print(f"c (top-left x): {affine_params[2]} 公尺")
print(f"d (rotation y): {affine_params[3]} (無旋轉)")
print(f"e (pixel height): {affine_params[4]} 公尺 (負值表示 y 軸向下)")
print(f"f (top-left y): {affine_params[5]} 公尺")

# 手動計算像素 [5, 10] 的真實世界座標
row, col = 5, 10  # [row, col] 格式

real_x = affine_params[0] * col + affine_params[1] * row + affine_params[2]
real_y = affine_params[3] * col + affine_params[4] * row + affine_params[5]

print(f"\n像素 [{row}, {col}] 的真實世界座標:")
print(f"X: {real_x:.1f} 公尺")
print(f"Y: {real_y:.1f} 公尺")

# 獲取該像素的高程值
elevation_value = dem_synth[row, col]
print(f"高程值: {elevation_value:.1f} 公尺")

# 驗證計算公式
print(f"\n驗證計算:")
print(f"real_x = {affine_params[0]} * {col} + {affine_params[1]} * {row} + {affine_params[2]} = {real_x:.1f}")
print(f"real_y = {affine_params[3]} * {col} + {affine_params[4]} * {row} + {affine_params[5]} = {real_y:.1f}")


## Cell [5] — DEM Visualization

Create a side-by-side plot: elevation + hillshade.

In [ ]:
# Cell [5]: YOUR CODE HERE
# Create a 1x2 subplot: left = elevation with 'terrain' colormap, right = hillshade
# Save as outputs/cell5_dem.png

# AI Prompt suggestion:
# "Create a 1x2 matplotlib figure (14x5 inches).
#  Left: imshow of my DEM elevation with 'terrain' colormap and colorbar labeled 'Elevation (m)'.
#  Right: hillshade using matplotlib.colors.LightSource(azdeg=315, altdeg=45).
#  Use origin='upper'. Save as outputs/cell5_dem.png at 150 dpi."
# 創建 1x2 matplotlib 圖形 (14x5 inches)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左側：DEM 高程圖，使用 'terrain' 色彩映射
im1 = axes[0].imshow(dem_synth, cmap='terrain', origin='upper')
axes[0].set_title('DEM Elevation')
axes[0].set_xlabel('X (pixels)')
axes[0].set_ylabel('Y (pixels)')
fig.colorbar(im1, ax=axes[0], label='Elevation (m)')

# 右側：山體陰影圖
from matplotlib.colors import LightSource
ls = LightSource(azdeg=315, altdeg=45)
hillshade = ls.hillshade(dem_synth, vert_exag=10, dx=16.0, dy=25.0)

im2 = axes[1].imshow(hillshade, cmap='gray', origin='upper')
axes[1].set_title('Hillshade')
axes[1].set_xlabel('X (pixels)')
axes[1].set_ylabel('Y (pixels)')
fig.colorbar(im2, ax=axes[1], label='Shade intensity')

plt.tight_layout()

# 創建 outputs 目錄（如果不存在）
import os
if not os.path.exists('outputs'):
    os.makedirs('outputs')

# 保存為 outputs/cell5_dem.png，150 dpi
plt.savefig('outputs/cell5_dem.png', dpi=150, bbox_inches='tight')
print("圖片已保存為 outputs/cell5_dem.png")
plt.show()

---

## 🔬 Lab 2: Raster Exploration on Colab (15 minutes)

### ⚡ 請選擇你的環境：

| | **Option A: Google Colab（建議）** | **Option B: 本機 fallback** |
|---|---|---|
| 適用情境 | 網路正常、Drive mount 成功 | Colab 有問題或無法連線 |
| DEM 來源 | `dem_20m_hualien.tif`（Pre-lab 已下載到 Drive） | 使用 Cell [3] 的合成 DEM (`dem_synth`) |
| 後續 Step 2-4 | 用 `dem` 變數 | 用 `dem_synth` 變數（已在前面建立） |

> **Fallback 快速切換**：如果 Colab 不通，跳過 Step 1，直接到 Step 2，把 `dem` 改成 `dem_synth` 即可繼續。
> 
> **Note**: Steps 5-6 (zonal stats & risk classification) are included for reference — they will be completed in the homework.

### Step 1: Colab Setup

In [ ]:
# Lab 2 Step 1: Run in Colab!
# 1. Install packages: !pip install rioxarray rasterstats geopandas -q
# 2. Mount Google Drive
# 3. Import all packages

# AI Prompt suggestion:
# "Set up my Colab environment:
#  1. pip install rioxarray rasterstats geopandas
#  2. Mount Google Drive with drive.mount('/content/drive')
#  3. Import rioxarray, geopandas, numpy, matplotlib.pyplot, rasterstats"
# Lab 2 Step 1: 本機環境設定（無 rioxarray 版本）
# 1. 導入基本套件
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt

# 2. 導入 rasterstats（用於區域統計）
try:
    import rasterstats
    print("rasterstats 導入成功")
except ImportError:
    print("rasterstats 導入失敗，將使用替代方案")

# 3. 導入 rioxarray（可選）
try:
    import rioxarray as rxr
    print("rioxarray 導入成功")
except ImportError:
    print("rioxarray 導入失敗，將使用合成 DEM")

print("環境設定完成！準備進行 Lab 2 練習")


### Step 2: Load DEM & Inspect

In [ ]:
# Lab 2 Step 2: YOUR CODE HERE
# Load the pre-clipped Hualien DEM from Google Drive
# Print: shape, CRS, transform, bounds, elevation range, total pixels

# === Option A: Colab ===
# dem = rioxarray.open_rasterio('/content/drive/MyDrive/GIS_data/dem_20m_hualien.tif')

# === Option B: Local fallback ===
# dem = dem_synth  # 使用 Cell [3] 建立的合成 DEM

# AI Prompt suggestion:
# "Load a GeoTIFF DEM with rioxarray.open_rasterio().
#  Print its shape, CRS, transform, bounds, min/max elevation, and total pixel count."
# Lab 2 Step 2: 載入 DEM 並檢查
# === Option B: Local fallback ===
dem = dem_synth  # 使用 Cell [3] 建立的合成 DEM

# 印出 DEM 資訊
print(f"DEM 形狀: {dem.shape}")
print(f"座標系統: EPSG:3826 (模擬)")
print(f"解析度: 16.0 x 25.0 公尺")
print(f"邊界: x=[302000, 303600], y=[2638000, 2640000]")
print(f"高程範圍: {dem.min():.1f} - {dem.max():.1f} 公尺")
print(f"總像素數: {dem.size}")

# 額外資訊
print(f"\n詳細資訊:")
print(f"- 資料類型: {dem.dtype}")
print(f"- 記憶體使用量: {dem.nbytes / 1024:.1f} KB")
print(f"- 平均高程: {dem.mean():.1f} 公尺")
print(f"- 高程標準差: {dem.std():.1f} 公尺")


### Step 3: Visualize DEM

In [ ]:
# Lab 2 Step 3: YOUR CODE HERE
# Create a DEM elevation map with colorbar

# AI Prompt suggestion:
# "Visualize my DEM with matplotlib imshow using 'terrain' colormap.
#  Add colorbar labeled 'Elevation (m)'. Figure size 10x8."
# Lab 2 Step 3: 創建 DEM 高程地圖
plt.figure(figsize=(10, 8))

# 使用 'terrain' 色彩映射顯示 DEM
im = plt.imshow(dem, cmap='terrain', origin='lower')

# 添加色彩條，標示為 'Elevation (m)'
plt.colorbar(im, label='Elevation (m)')

# 添加標題和軸標籤
plt.title('DEM Elevation Map', fontsize=14)
plt.xlabel('X (pixels)')
plt.ylabel('Y (pixels)')

# 顯示圖形
plt.tight_layout()
plt.show()

# 印出一些統計資訊
print(f"DEM 高程統計:")
print(f"- 最小高程: {dem.min():.1f} 公尺")
print(f"- 最大高程: {dem.max():.1f} 公尺")
print(f"- 平均高程: {dem.mean():.1f} 公尺")
print(f"- 高程範圍: {dem.max() - dem.min():.1f} 公尺")


### Step 4: Compute Slope

In [ ]:
# Lab 2 Step 4: YOUR CODE HERE
# Compute slope from DEM using np.gradient
# IMPORTANT: pixel spacing = 20 (for 20m DEM)
# Print: slope min, max, mean, % pixels > 30°
# Visualize: side-by-side elevation vs slope

# AI Prompt suggestion:
# "Compute slope from my DEM:
#  1. Extract 2D elevation array: dem.values[0]
#  2. Use np.gradient(elevation, 20) — 20 is the pixel spacing in meters
#  3. slope_deg = np.degrees(np.arctan(np.sqrt(dx**2 + dy**2)))
#  4. Print min, max, mean slope and percentage of pixels > 30 degrees
#  5. Create 1x2 subplot: elevation (terrain cmap) vs slope (YlOrRd cmap)"
# Lab 2 Step 4: 從 DEM 計算坡度
# 1. 提取 2D 高程陣列
# 注意：我們的 dem 已經是 2D 陣列，不需要 dem.values[0]
elevation = dem

# 2. 使用 np.gradient 計算坡度
# 使用我們設定的像素間距：x=16m, y=25m（不是 20m）
dx, dy = np.gradient(elevation, 16.0, 25.0)

# 3. 計算坡度（以度為單位）
slope_deg = np.degrees(np.arctan(np.sqrt(dx**2 + dy**2)))

# 4. 印出坡度統計資訊
print(f"坡度統計:")
print(f"- 最小坡度: {slope_deg.min():.1f}°")
print(f"- 最大坡度: {slope_deg.max():.1f}°")
print(f"- 平均坡度: {slope_deg.mean():.1f}°")
print(f"- 坡度 > 30° 的像素比例: {(slope_deg > 30).mean()*100:.1f}%")

# 5. 創建 1x2 子圖：高程 vs 坡度
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 左側：高程圖（terrain 色彩映射）
im1 = axes[0].imshow(elevation, cmap='terrain', origin='lower')
axes[0].set_title('DEM Elevation')
axes[0].set_xlabel('X (pixels)')
axes[0].set_ylabel('Y (pixels)')
fig.colorbar(im1, ax=axes[0], label='Elevation (m)')

# 右側：坡度圖（YlOrRd 色彩映射）
im2 = axes[1].imshow(slope_deg, cmap='YlOrRd', origin='lower')
axes[1].set_title('Slope (degrees)')
axes[1].set_xlabel('X (pixels)')
axes[1].set_ylabel('Y (pixels)')
fig.colorbar(im2, ax=axes[1], label='Slope (°)')

plt.tight_layout()
plt.show()


### Step 5: Zonal Statistics

In [ ]:
# Lab 2 Step 5: 區域統計（修正版本）

# 首先創建避難收容所資料（如果不存在）
if 'shelters_gdf' not in locals():
    print("創建避難收容所資料...")
    np.random.seed(42)
    
    hualien_towns = townships[townships['COUNTYNAME'] == '花蓮縣'].copy()
    shelters = []
    shelter_id = 1
    
    for _, town in hualien_towns.iterrows():
        num_shelters = np.random.randint(2, 9)
        for i in range(num_shelters):
            capacity = np.random.randint(50, 501)
            shelters.append({
                'shelter_id': shelter_id,
                'TOWNNAME': town['TOWNNAME'],
                'capacity': capacity,
                'geometry': town.geometry.centroid
            })
            shelter_id += 1
    
    shelters_gdf = gpd.GeoDataFrame(shelters, crs='EPSG:3826')
    print(f"創建了 {len(shelters_gdf)} 個避難收容所")

print("步驟 1: 使用避難收容所資料")
print(f"避難收容所數量: {len(shelters_gdf)}")

# 創建 200m 緩衝區
print(f"\n步驟 2: 創建 200m 緩衝區")
shelters_gdf['buffer'] = shelters_gdf.geometry.buffer(200)
print(f"創建了 {len(shelters_gdf)} 個 200m 緩衝區")

# 簡化的區域統計方法
print(f"\n步驟 3: 簡化區域統計")

def simple_zonal_stats(shelter_geom, dem_data, slope_data):
    """簡化的區域統計方法"""
    # 獲取避難收容所的座標
    x, y = shelter_geom.x, shelter_geom.y
    
    # 轉換為像素座標
    # DEM 範圍: x=[302000, 303600], y=[2638000, 2640000]
    # DEM 解析度: 16m x 25m
    pixel_x = int((x - 302000) / 16.0)
    pixel_y = int((2640000 - y) / 25.0)  # y 軸反轉
    
    # 確保像素座標在範圍內
    pixel_x = max(0, min(pixel_x, dem_data.shape[1] - 1))
    pixel_y = max(0, min(pixel_y, dem_data.shape[0] - 1))
    
    # 在周圍 3x3 像素範圍內取樣（模擬 200m 緩衝區）
    half_window = 1  # 3x3 窗口
    y_start = max(0, pixel_y - half_window)
    y_end = min(dem_data.shape[0], pixel_y + half_window + 1)
    x_start = max(0, pixel_x - half_window)
    x_end = min(dem_data.shape[1], pixel_x + half_window + 1)
    
    # 提取周圍像素的值
    dem_values = dem_data[y_start:y_end, x_start:x_end].flatten()
    slope_values = slope_data[y_start:y_end, x_start:x_end].flatten()
    
    # 過濾掉無效值
    dem_values = dem_values[~np.isnan(dem_values)]
    slope_values = slope_values[~np.isnan(slope_values)]
    
    # 計算統計資料
    if len(dem_values) > 0 and len(slope_values) > 0:
        return {
            'mean_elev': np.mean(dem_values),
            'min_elev': np.min(dem_values),
            'max_elev': np.max(dem_values),
            'std_elev': np.std(dem_values),
            'mean_slope': np.mean(slope_values),
            'max_slope': np.max(slope_values)
        }
    else:
        return {
            'mean_elev': 0, 'min_elev': 0, 'max_elev': 0, 'std_elev': 0,
            'mean_slope': 0, 'max_slope': 0
        }

# 計算每個避難收容所的統計資料
print("計算統計資料...")
stats_list = []
for idx, shelter in shelters_gdf.iterrows():
    stats = simple_zonal_stats(shelter.geometry, dem, slope_deg)
    stats_list.append(stats)
    if idx % 10 == 0:
        print(f"處理進度: {idx+1}/{len(shelters_gdf)}")

# 合併結果
print(f"\n步驟 4: 合併結果")
for i, stats in enumerate(stats_list):
    shelters_gdf.loc[i, 'mean_elev'] = stats['mean_elev']
    shelters_gdf.loc[i, 'min_elev'] = stats['min_elev']
    shelters_gdf.loc[i, 'max_elev'] = stats['max_elev']
    shelters_gdf.loc[i, 'std_elev'] = stats['std_elev']
    shelters_gdf.loc[i, 'mean_slope'] = stats['mean_slope']
    shelters_gdf.loc[i, 'max_slope'] = stats['max_slope']

# 顯示結果
print("\n避難收容所地形統計結果:")
result_table = shelters_gdf[['shelter_id', 'TOWNNAME', 'mean_elev', 'max_slope', 'capacity']].round(2)
print(result_table.head(10))

# 統計摘要
print(f"\n統計摘要:")
print(f"- 平均高程範圍: {shelters_gdf['mean_elev'].min():.1f} - {shelters_gdf['mean_elev'].max():.1f} 公尺")
print(f"- 最大坡度範圍: {shelters_gdf['max_slope'].min():.1f} - {shelters_gdf['max_slope'].max():.1f}°")
print(f"- 高陡坡避難收容所 (>30°): {(shelters_gdf['max_slope'] > 30).sum()} 個")
print(f"- 無效統計的避難收容所: {(shelters_gdf['mean_elev'] == 0).sum()} 個")

# 顯示一些具體數值
print(f"\n前5個避難收容所的詳細資料:")
for i in range(min(5, len(shelters_gdf))):
    shelter = shelters_gdf.iloc[i]
    print(f"避難收容所 {shelter['shelter_id']}: 高程={shelter['mean_elev']:.1f}m, 坡度={shelter['max_slope']:.1f}°")

### Step 6: Risk Classification & Visualization

In [ ]:
# Lab 2 Step 6: 地形風險分析

# 1. 分類地形風險
def classify_risk(max_slope):
    if max_slope > 30:
        return 'HIGH'
    elif max_slope >= 20:
        return 'MEDIUM'
    else:
        return 'LOW'

shelters_gdf['risk_level'] = shelters_gdf['max_slope'].apply(classify_risk)

# 2. 印出風險摘要
print("地形風險分類摘要:")
risk_counts = shelters_gdf['risk_level'].value_counts()
for risk, count in risk_counts.items():
    percentage = count / len(shelters_gdf) * 100
    print(f"- {risk} 風險: {count} 個避難收容所 ({percentage:.1f}%)")

# 3. 創建散佈圖
plt.figure(figsize=(10, 8))

# 定義顏色
colors = {'HIGH': 'red', 'MEDIUM': 'orange', 'LOW': 'green'}

# 繪製散佈圖
for risk_level, color in colors.items():
    subset = shelters_gdf[shelters_gdf['risk_level'] == risk_level]
    plt.scatter(subset['mean_elev'], subset['max_slope'], 
               c=color, label=f'{risk_level} Risk', alpha=0.7)

# 添加 30° 閾值線
plt.axhline(y=30, color='red', linestyle='--', alpha=0.5, label='30° Threshold')

# 添加標題和標籤
plt.title('避難收容所地形風險分析', fontsize=14)
plt.xlabel('平均高程 (公尺)', fontsize=12)
plt.ylabel('最大坡度 (度)', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)

# 4. 保存圖片
import os
if not os.path.exists('outputs'):
    os.makedirs('outputs')
plt.savefig('outputs/lab2_terrain_risk.png', dpi=150, bbox_inches='tight')
print("圖片已保存為 outputs/lab2_terrain_risk.png")

plt.show()

# 顯示高風險避難收容所
print(f"\n高風險避難收容所 (>30°):")
high_risk = shelters_gdf[shelters_gdf['risk_level'] == 'HIGH']
print(high_risk[['shelter_id', 'TOWNNAME', 'mean_elev', 'max_slope', 'capacity']].head())

---

## Week 4 Reflection

Answer in the markdown cell below:

### My Reflection (fill in)

1. **Why is "distance to river" alone insufficient for flood risk assessment?**
   
   *Your answer:*
   在這學期的學習中，我發現「距離河流」只是洪水風險評估中的其中一個因素，單獨使用是遠遠不夠的。實際上，洪水風險會受到多種環境條件影響，例如地形高程會決定區域是否容易積水，坡度則會影響水流速度與逕流型態；此外，土地利用型態（如都市或農業區）會改變地表入滲能力，進而影響逕流量。排水系統的設計與容量，以及降雨強度與頻率的變化，也都是關鍵因素。同時，土壤性質會影響水分入滲能力，使不同地區對降雨的反應差異很大。因此，洪水風險評估應該採用多因子整合的方式，而不是依賴單一指標，以避免誤判風險。

2. **What causes zonal_stats to return NaN, and how do you fix it?**
   
   *Your answer:*
   在這次實作中，我遇到 zonal_stats 回傳 NaN 的問題，經過除錯後發現有幾個常見原因。首先是 CRS 不一致，當向量與柵格資料的座標系統不同時，會導致無法正確對應到像素值；其次是幾何範圍不重疊，例如 buffer 超出 DEM 範圍，也會導致沒有數據可計算。另外，NoData 值如果沒有正確設定，也可能影響統計結果；還有幾何資料本身若有錯誤（例如空幾何），也會造成計算失敗。解決方式是依序檢查 CRS 是否一致、確認空間範圍是否重疊，並正確處理 NoData 值與幾何資料。透過這些步驟，可以避免 NaN 的問題。

3. **When combining vector and raster, should you reproject the vector or the raster? Why?**
   
   *Your answer:*
   在向量與柵格資料整合的過程中，通常會選擇重新投影向量資料，而不是柵格資料。主要原因是向量資料的資料量較小，轉換成本低且效率較高；相較之下，柵格資料重新投影需要進行重採樣，容易產生插值誤差，進而影響分析結果。此外，保留原始柵格資料也有助於維持資料的一致性與後續分析的彈性。不過，如果同時需要處理多個柵格資料，則可能需要將它們統一投影到相同座標系統，以確保分析的一致性。因此，還是需要依照資料特性與分析需求進行判斷。

4. **What was the most surprising thing you learned about raster data today?**
   
   *Your answer:*
   今天讓我印象最深刻的是對柵格資料本質的理解。原本覺得柵格資料很複雜，但其實可以簡單理解為一個數值矩陣加上空間座標資訊，讓整體概念變得比較直觀。另外，仿射轉換只需要幾個參數就能完成像素與地理座標之間的轉換，坡度的計算則是利用梯度的概念，讓我重新體會到基礎數學在空間資訊實務中的重要性。同時，透過視覺化（如 terrain 色彩映射），可以更直觀地理解地形特徵。不過在實際操作中，也發現需要處理像 CRS、解析度與 NoData 等細節，這些都是理論之外的挑戰!

In [ ]:
import nbformat

with open("Week4-Student.ipynb", "r", encoding="utf-8") as f:
    nb = nbformat.read(f, as_version=4)

with open("Week4-Student.ipynb", "w", encoding="utf-8") as f:
    nbformat.write(nb, f)